# Sesión 17 · Forma: sesgo, curtosis, Chebyshev y regla empírica

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno de ejecución
→ **R**, y después Archivo → Guardar una copia en Drive.

## 1 · Los datos

Tres bases: los hogares de la ENIGH 2024, los precios diarios de Apple (AAPL) y
Tesla (TSLA), y una nueva, `economias`: la esperanza de vida y el acceso a
electricidad de todas las economías del mundo, según el Banco Mundial. Cada
renglón de `economias` es un país o territorio.

In [ ]:
library(tidyverse)
source("https://raw.githubusercontent.com/cjjmdata/analisis_datos_i/main/R/colores.R")   # colores del curso

hogares   <- read_csv("https://raw.githubusercontent.com/cjjmdata/analisis_datos_i/main/datos/enigh2024_hogares.csv.gz", show_col_types = FALSE)
mercado   <- read_csv("https://raw.githubusercontent.com/cjjmdata/analisis_datos_i/main/datos/mercado_diario.csv", show_col_types = FALSE)
economias <- read_csv("https://raw.githubusercontent.com/cjjmdata/analisis_datos_i/main/datos/banco_mundial_economias.csv", show_col_types = FALSE)

glimpse(economias)

## 2 · Tres formas

El porcentaje del gasto que cada hogar dedica a alimentos, el gasto del hogar y
la esperanza de vida de cada economía. Antes de correr la celda: ¿cuál crees que
tiene una cola larga, y hacia qué lado?

In [ ]:
con_gasto <- filter(hogares, gasto > 0)          # sin gasto no hay porcentaje
nrow(hogares) - nrow(con_gasto)                     # hogares que se descartan

alimentos <- 100 * con_gasto$alimentos / con_gasto$gasto

ggplot(tibble(x = alimentos), aes(x)) +
  geom_histogram(binwidth = 2.5, fill = TEAL_500, color = "white") +
  geom_vline(xintercept = mean(alimentos), color = col_media, linewidth = 1.2) +
  geom_vline(xintercept = median(alimentos), color = col_mediana, linewidth = 1.2, linetype = "dashed") +
  labs(x = "% del gasto en alimentos", y = "Frecuencia")

In [ ]:
ggplot(hogares, aes(x = gasto)) +
  geom_histogram(binwidth = 5000, fill = TEAL_500, color = "white") +
  geom_vline(xintercept = mean(hogares$gasto), color = col_media, linewidth = 1.2) +
  geom_vline(xintercept = median(hogares$gasto), color = col_mediana, linewidth = 1.2, linetype = "dashed") +
  coord_cartesian(xlim = c(0, 200000)) +
  labs(x = "Gasto trimestral del hogar, pesos", y = "Frecuencia")

In [ ]:
ggplot(economias, aes(x = esperanza_vida)) +
  geom_histogram(binwidth = 2, fill = TEAL_500, color = "white") +
  geom_vline(xintercept = mean(economias$esperanza_vida), color = col_media, linewidth = 1.2) +
  geom_vline(xintercept = median(economias$esperanza_vida), color = col_mediana, linewidth = 1.2, linetype = "dashed") +
  labs(x = "Esperanza de vida al nacer, años", y = "Frecuencia")

La línea continua es la media y la discontinua la mediana. ¿Hacia qué lado se
mueve la media en cada caso?

### La media que no representa a nadie

La media mundial del acceso a electricidad, ¿a cuántas economías describe? Dos
economías no tienen dato y se quitan antes de calcular.

In [ ]:
luz <- economias$acceso_electricidad[!is.na(economias$acceso_electricidad)]
sum(is.na(economias$acceso_electricidad))           # economías sin dato

mean(luz)
median(luz)

sum(abs(luz - mean(luz)) <= 5)                      # a menos de 5 puntos de la media
sum(luz == 100)                                     # con acceso total

## 3 · Coeficiente de asimetría

Los integrantes de los cinco primeros hogares. Con tu calculadora: la media, las
desviaciones, sus cuadrados y sus cubos.

$$g_1 = \frac{\frac{1}{n}\sum (x_i - \bar{x})^3}{\left(\frac{1}{n}\sum (x_i - \bar{x})^2\right)^{3/2}}$$

In [ ]:
cinco <- hogares$integrantes[1:5]
cinco

d <- cinco - mean(cinco)
d
d^2
d^3

mean(d^3) / mean(d^2)^(3/2)     # la fórmula, traducida literalmente

### Una función propia

R no trae una función para la asimetría. La escribes una vez y la usas como
`mean()` o `sd()`.

In [ ]:
asimetria <- function(x) {
  d <- x - mean(x)                  # las desviaciones
  mean(d^3) / mean(d^2)^(3/2)       # la última línea es el resultado
}

asimetria(cinco)

¿Coinciden los signos con lo que viste en los histogramas? ¿Qué pasa si le
pasas una columna con valores faltantes?

In [ ]:
asimetria(alimentos)
asimetria(hogares$gasto)
asimetria(economias$esperanza_vida)

asimetria(economias$acceso_electricidad)
asimetria(luz)

### Cinco hogares contra todos

Los integrantes de todos los hogares están sesgados a la derecha. ¿Qué parte de
las muestras de cinco hogares crees que da un sesgo negativo o cero?
`replicate()` repite una instrucción las veces que le pidas.

In [ ]:
asimetria(hogares$integrantes)

muestras <- replicate(5000, asimetria(sample(hogares$integrantes, 5)))

sum(is.nan(muestras))                               # cinco hogares iguales: 0 entre 0
mean(muestras[!is.nan(muestras)] <= 0)

Cambia el 5 por 30 y por 300 en `sample()`. ¿Qué pasa con la proporción?

### Un hogar mueve el coeficiente

In [ ]:
sin_mayor <- hogares$ingreso[hogares$ingreso < max(hogares$ingreso)]

asimetria(hogares$ingreso)
asimetria(sin_mayor)

## 4 · Curtosis

Los rendimientos diarios de AAPL, en porcentaje, contra una curva normal con la
misma media y la misma desviación estándar. ¿En qué zonas hay más días de los que
predice la curva?

In [ ]:
rendimientos <- mercado |>
  arrange(clave, fecha) |>
  group_by(clave) |>
  mutate(rend = (ajustado / lag(ajustado) - 1) * 100) |>
  ungroup() |>
  filter(!is.na(rend))                              # el primer día no tiene rendimiento

aapl <- rendimientos$rend[rendimientos$clave == "AAPL"]

ggplot(tibble(x = aapl), aes(x)) +
  geom_histogram(aes(y = after_stat(density)), binwidth = 0.25, fill = TEAL_500, color = "white") +
  stat_function(fun = dnorm, args = list(mean = mean(aapl), sd = sd(aapl)),
                color = col_destaca, linewidth = 1.2) +
  coord_cartesian(xlim = c(-8, 8)) +
  labs(x = "Rendimiento diario de AAPL, %", y = "Densidad")

### Primero a mano

$$g_2 = \frac{\frac{1}{n}\sum (x_i - \bar{x})^4}{\left(\frac{1}{n}\sum (x_i - \bar{x})^2\right)^{2}} - 3$$

Se resta 3 porque una normal da 3: el resultado es el **exceso de curtosis**.

In [ ]:
mean(d^4) / mean(d^2)^2 - 3     # los cinco hogares, a mano

curtosis <- function(x) {
  d <- x - mean(x)
  mean(d^4) / mean(d^2)^2 - 3       # 0 para una normal
}

curtosis(cinco)
curtosis(aapl)

¿Qué parte de la curtosis de AAPL viene de los días a más de 3 desviaciones
estándar de la media?

In [ ]:
z <- (aapl - mean(aapl)) / sd(aapl)

sum(abs(z) > 3)                                     # días a más de 3 desviaciones
mean(abs(z) > 3)                                    # como proporción de los días

sum(z[abs(z) > 3]^4) / sum(z^4)                     # su parte de la suma de cuartas potencias

## 5 · Chebyshev y la regla empírica

Para cualquier conjunto de datos, al menos $1 - 1/k^2$ de los datos está a menos
de $k$ desviaciones estándar de la media.

In [ ]:
k <- c(1.5, 2, 3)
1 - 1 / k^2

El gasto de los hogares: ¿cuál es el intervalo $\bar{x} \pm 2s$, y qué
porcentaje de los hogares cae ahí?

In [ ]:
mean(hogares$gasto) - 2 * sd(hogares$gasto)
mean(hogares$gasto) + 2 * sd(hogares$gasto)

mean(abs(hogares$gasto - mean(hogares$gasto)) <= 2 * sd(hogares$gasto))

La regla empírica dice 68%, 95% y 99.7% para datos de forma normal. Compárala
con lo que observas en cada variable.

In [ ]:
# Proporción de datos a menos de 1, 2 y 3 desviaciones estándar de la media
dentro <- function(x) {
  c(k1 = mean(abs(x - mean(x)) <= 1 * sd(x)),
    k2 = mean(abs(x - mean(x)) <= 2 * sd(x)),
    k3 = mean(abs(x - mean(x)) <= 3 * sd(x)))
}

dentro(alimentos)
dentro(hogares$gasto)
dentro(aapl)
dentro(economias$esperanza_vida)
dentro(luz)

¿En cuál de las cinco acierta la regla empírica? ¿Se cumple Chebyshev en todas?

## 6 · Tu serie

Sobre la variable numérica de tu serie del portafolio: su histograma con media y
mediana, `asimetria()`, `curtosis()` y la proporción de datos a menos de 1, 2 y 3
desviaciones estándar. ¿Cuál regla la describe mejor?

In [ ]:
# Lectura de tu serie
